# ⚡ Module 09 — Iterators, Generators, Decorators & Context Managers

## 🎯 What you will learn
1. The iterator protocol (`__iter__`, `__next__`) — how `for` really works
2. Generators (`yield`), generator expressions, lazy evaluation, `yield from`, `send()`
3. Closures
4. Decorators (simple, with arguments, stacking, `functools.wraps`, class decorators)
5. Context managers and the `with` statement
6. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐ (advanced-level papers love decorators & generators)

---
# PART A — ITERATORS

- **Iterable**: an object you can loop over (has `__iter__`) — list, str, dict, file…
- **Iterator**: an object that produces the next value each time you call `next()` (has `__next__`) and raises `StopIteration` when finished.

In [1]:
nums = [10, 20, 30]
it = iter(nums)          # get an iterator from an iterable
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration: no more items")

print(iter(nums) is nums, iter(it) is it)   # iterators return themselves from iter()

10 20 30
StopIteration: no more items
False True


### Build your own iterator

In [2]:
class Countdown:
    def __init__(self, start):
        self.current = start
    def __iter__(self):
        return self
    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

for n in Countdown(4):
    print(n, end=" ")
print()
print(list(Countdown(3)), sum(Countdown(4)))

4 3 2 1 
[3, 2, 1] 10


### Iterators are **single-use** and **lazy**

In [3]:
it = iter([1, 2, 3])
print(list(it))
print(list(it))          # empty! already consumed

r = range(3)
print(list(r), list(r))  # range is an iterable, NOT an iterator: reusable

[1, 2, 3]
[]
[0, 1, 2] [0, 1, 2]


---
# PART B — GENERATORS

A **generator function** contains `yield`. Calling it does **not** run the body; it returns a generator (an iterator). Each `next()` runs the body until the next `yield`, **pauses**, and remembers all local state.

In [4]:
def count_up_to(n):
    print("  (starting)")
    i = 1
    while i <= n:
        yield i           # pause here and hand out i
        i += 1
    print("  (finished)")

gen = count_up_to(3)
print(type(gen))
print(next(gen))
print(next(gen))
print(next(gen))
try:
    next(gen)
except StopIteration:
    print("done")

for x in count_up_to(2):
    print("got", x)

<class 'generator'>
  (starting)
1
2
3
  (finished)
done
  (starting)
got 1
got 2
  (finished)


### Why generators? Memory & laziness

In [5]:
import sys
big_list = [i * i for i in range(1_000_000)]
big_gen = (i * i for i in range(1_000_000))       # generator EXPRESSION uses ( )
print("list :", sys.getsizeof(big_list), "bytes")
print("gen  :", sys.getsizeof(big_gen), "bytes")
print(sum(big_gen))

# Infinite generator - impossible with a list!
def naturals():
    n = 1
    while True:
        yield n
        n += 1

from itertools import islice
print(list(islice(naturals(), 5)))

def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b
print(list(islice(fibonacci(), 12)))

list : 8448728 bytes
gen  : 200 bytes
333332833333500000
[1, 2, 3, 4, 5]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Pipelines: chain generators like Unix pipes

In [6]:
def read_numbers(text):
    for token in text.split():
        yield int(token)

def only_even(nums):
    for n in nums:
        if n % 2 == 0:
            yield n

def squared(nums):
    for n in nums:
        yield n * n

pipeline = squared(only_even(read_numbers("1 2 3 4 5 6")))
print(list(pipeline))

[4, 16, 36]


### `yield from`, `send()`

In [7]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)     # delegate to a sub-generator
        else:
            yield item
print(list(flatten([1, [2, [3, 4]], 5])))

def running_average():
    total, count, avg = 0, 0, None
    while True:
        value = yield avg                # receives a value from send()
        total += value
        count += 1
        avg = total / count

g = running_average()
next(g)                                  # prime the generator
print(g.send(10), g.send(20), g.send(60))
g.close()

[1, 2, 3, 4, 5]
10.0 15.0 30.0


---
# PART C — CLOSURES
A **closure** is an inner function that remembers variables from its enclosing function even after that function has finished.

In [8]:
def make_power(exp):
    def power(base):
        return base ** exp        # `exp` is remembered
    return power

square, cube = make_power(2), make_power(3)
print(square(5), cube(5))
print(square.__closure__[0].cell_contents)

def counter():
    n = 0
    def step():
        nonlocal n
        n += 1
        return n
    return step
c = counter()
print(c(), c(), c())

25 125
2
1 2 3


---
# PART D — DECORATORS

A **decorator** is a function that takes a function and returns an enhanced function. `@decorator` is just syntax sugar for `func = decorator(func)`.

In [9]:
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper() + "!!!"
    return wrapper

@shout
def greet(name):
    return f"hello {name}"

print(greet("asha"))
# same as: greet = shout(greet)

HELLO ASHA!!!


### A proper decorator uses `functools.wraps` (keeps the original name & docstring)

In [10]:
import functools, time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Add numbers up to n."""
    return sum(range(n))

print(slow_sum(1_000_00))
print(slow_sum.__name__, "|", slow_sum.__doc__)      # preserved thanks to @wraps

[timer] slow_sum took 0.003038s
4999950000
slow_sum | Add numbers up to n.


### Decorators with arguments (three levels of nesting)

In [11]:
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say(msg):
    print(msg)

say("hi")

hi
hi
hi


### Practical decorators

In [12]:
# 1. logging
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}{args} {kwargs}")
        return func(*args, **kwargs)
    return wrapper

# 2. access check
def require_admin(func):
    @functools.wraps(func)
    def wrapper(user, *args, **kwargs):
        if user.get("role") != "admin":
            raise PermissionError("admins only")
        return func(user, *args, **kwargs)
    return wrapper

# 3. retry
def retry(times=3):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*a, **k):
            for attempt in range(1, times + 1):
                try:
                    return func(*a, **k)
                except Exception as e:
                    print(f"attempt {attempt} failed: {e}")
            raise RuntimeError("all attempts failed")
        return wrapper
    return deco

# 4. memoisation
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@log_calls
def add(a, b): return a + b

@require_admin
def delete_all(user): return "deleted"

calls = {"n": 0}
@retry(3)
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ValueError("boom")
    return "success"

@memoize
def slow_square(n):
    print("  computing", n)
    return n * n

print(add(2, 3))
print(delete_all({"role": "admin"}))
try:
    delete_all({"role": "guest"})
except PermissionError as e:
    print("PermissionError:", e)
print(flaky())
print(slow_square(4), slow_square(4))

calling add(2, 3) {}
5
deleted
PermissionError: admins only
attempt 1 failed: boom
attempt 2 failed: boom
success
  computing 4
16 16


### Stacking decorators — applied bottom-up

In [13]:
def bold(f):
    return lambda *a, **k: f"<b>{f(*a, **k)}</b>"
def italic(f):
    return lambda *a, **k: f"<i>{f(*a, **k)}</i>"

@bold
@italic
def text(): return "hi"
print(text())     # <b><i>hi</i></b>   (italic applied first, then bold)

<b><i>hi</i></b>


### Class-based decorators and built-in decorators

In [14]:
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.count = 0
    def __call__(self, *a, **k):
        self.count += 1
        return self.func(*a, **k)

@CountCalls
def ping(): return "pong"
ping(); ping()
print(ping.count)

class Temperature:
    def __init__(self, c): self._c = c
    @property                       # built-in decorator: method -> attribute
    def fahrenheit(self): return self._c * 9 / 5 + 32
    @staticmethod                   # no self/cls
    def is_hot(c): return c > 35
    @classmethod                    # receives the class
    def from_f(cls, f): return cls((f - 32) * 5 / 9)

print(Temperature(100).fahrenheit, Temperature.is_hot(40), Temperature.from_f(212)._c)

2
212.0 True 100.0


---
# PART E — CONTEXT MANAGERS & `with`

`with` guarantees setup **and** cleanup (even if an exception happens). Files, locks, DB connections all use it.

In [15]:
# Class-based context manager
class Timer:
    def __enter__(self):
        import time
        self.t0 = time.perf_counter()
        print("enter")
        return self                      # bound to the `as` variable
    def __exit__(self, exc_type, exc, tb):
        import time
        print(f"exit (took {time.perf_counter() - self.t0:.4f}s); exception: {exc_type}")
        return True                      # True => suppress the exception

with Timer() as t:
    print("inside")
    1 / 0                                # suppressed by __exit__
print("after with")

# Generator-based context manager (shortcut)
from contextlib import contextmanager, suppress, redirect_stdout
import io

@contextmanager
def tag(name):
    print(f"<{name}>", end="")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("p"):
    print("text", end="")

with suppress(FileNotFoundError):        # ignore a specific error
    open("nope.txt")

buf = io.StringIO()
with redirect_stdout(buf):
    print("captured!")
print(repr(buf.getvalue()))

enter
inside
exit (took 0.0000s); exception: <class 'ZeroDivisionError'>
after with
<p>text</p>
'captured!\n'


---
# 🕵️ Hidden Facts & Pro Tricks

In [16]:
# 1. A generator remembers state; you can inspect it
def g():
    yield 1; yield 2
gen = g()
import inspect
print(inspect.getgeneratorstate(gen)); next(gen); print(inspect.getgeneratorstate(gen))

# 2. Generator expression as the only argument needs no extra parentheses
print(sum(x * x for x in range(5)), max(len(w) for w in ["a", "abc"]))

# 3. `next(gen, default)`: get first match without exceptions
print(next((x for x in range(100) if x % 7 == 0 and x > 20), None))

# 4. `iter(callable, sentinel)`: read a file in blocks until empty
import io
f = io.BytesIO(b"abcdefg")
print(list(iter(lambda: f.read(3), b"")))

# 5. `enumerate`, `zip`, `map`, `filter`, `reversed` are all lazy iterators
m = map(str, range(3))
print(m, list(m))

# 6. `itertools.tee` duplicates an iterator; `cycle` repeats forever
import itertools
a, b = itertools.tee(iter([1, 2, 3]))
print(list(a), list(b))
print(list(itertools.islice(itertools.cycle("ab"), 5)))

# 7. A decorator can be applied manually and can also register things
registry = []
def register(f):
    registry.append(f.__name__)
    return f
@register
def alpha(): pass
@register
def beta(): pass
print(registry)

# 8. @functools.lru_cache(maxsize=...) and @functools.cache give memoisation in one line
# 9. @functools.singledispatch gives overloading by argument type
from functools import singledispatch
@singledispatch
def show(x): return f"object {x}"
@show.register
def _(x: int): return f"int {x}"
@show.register
def _(x: list): return f"list of {len(x)}"
print(show(1), show([1, 2]), show("s"))

# 10. @dataclass, @property, @staticmethod, @classmethod, @abstractmethod: all decorators
# 11. Generators can be used as coroutines - the ancestor of async/await
# 12. `yield` is also an expression: x = yield value

GEN_CREATED
GEN_SUSPENDED
30 3
21
[b'abc', b'def', b'g']
<map object at 0x00000263BBE26020> ['0', '1', '2']
[1, 2, 3] [1, 2, 3]
['a', 'b', 'a', 'b', 'a']
['alpha', 'beta']
int 1 list of 2 object s


**Hidden fact:** Every `for` loop, comprehension, `sum()`, `list()`, `sorted()` and unpacking works on *any* iterator — so writing your own generator instantly makes your data work with the whole language.

**Hidden fact:** a decorator runs **at definition time** (when Python reads the `def`), while the wrapper runs at **call time**. Print statements in the outer function appear at import!

**Hidden fact:** `@wraps` matters for debugging tools, `help()`, and frameworks like Flask/FastAPI which look up functions by `__name__`.

---
# 📝 Exam Corner

**Theory**
1. Difference between iterable and iterator. What is `StopIteration`?
2. What is a generator? Difference between `return` and `yield`; generator vs list.
3. What are decorators? Write one that measures time.
4. What is a closure?
5. What does `with` do? Write `__enter__`/`__exit__`.
6. Generator expression vs list comprehension.

**Output prediction**

In [17]:
def gen():
    for i in range(3):
        x = yield i
        print("got", x)

g = gen()
print(next(g))
print(g.send("A"))
print(next(g))

def deco(f):
    print("decorating", f.__name__)
    def w(*a):
        print("before")
        return f(*a)
    return w

@deco
def hi(): print("hi")
print("defined")
hi()

0
got A
1
got None
2
decorating hi
defined
before
hi


<details><summary>💡 Answers</summary>

```
0
got A
1
got None
2
decorating hi
defined
before
hi
```
`send("A")` resumes the generator (`x = "A"`), which prints and yields 1. Decoration happens at definition time, before "defined".
</details>

---
## ✅ Summary
- `iter()` + `next()` power every `for`. Iterators are single-use.
- `yield` turns a function into a lazy generator: tiny memory, infinite sequences, pipelines.
- Closures capture variables; decorators wrap functions (`@wraps`, arguments via 3 nested levels).
- `with` = guaranteed cleanup through `__enter__/__exit__` or `@contextmanager`.

➡️ **Next:** Module 10 — Modules, Packages & the Standard Library.